In [2]:
rank_values = {
    "2": 2, "3": 3, "4": 4, "5": 5, "6": 6, "7": 7, "8": 8, "9": 9, "10": 10,
    "J": 10, "Q": 10, "K": 10, "A": 11,
}

print(rank_values)

{'2': 2, '3': 3, '4': 4, '5': 5, '6': 6, '7': 7, '8': 8, '9': 9, '10': 10, 'J': 10, 'Q': 10, 'K': 10, 'A': 11}


In [45]:
def hand_values(cards):
    total = 0
    aces = 0 

    for card in cards:
        total += rank_values[card]
        if card == "A":
            aces += 1

    while total > 21 and aces > 0:
        total -= 10
        aces -= 1
    return total

print(hand_values("9"))

9


In [46]:
def parse_state(text):
    parts = text.split("|")
    cleaned_parts = []
    
    for part in parts:
        cleaned_parts.append(part.strip())
    
    hand_str, dealer_upcard, flag = cleaned_parts
    
    hand = []
    for rank in hand_str.split(","):
        hand.append(rank.strip())
    print("Hand =", hand)
    return hand, dealer_upcard, flag

print(parse_state("10, 6 | 9 | first"))

Hand = ['10', '6']
(['10', '6'], '9', 'first')


### Data Model

In [8]:
state = {
    "hand" : ["10", "6"],
    "dealer" : "9",
    "first" : True,
    "total" : 16,
    "busted" : False
}

### Implement Parse_state

##### Example - "10, 6 | 9 | first"
- If "10 , 6 | 9 | first":
- The player's first hand : 10, 6 will be split into a list ["10", "6"]
- Return the dealer's hand
- Insert an if statement to evaluate if first decision
- Return the total of the player'shand
- Evaluate if > 21

### Generate Actions

- If First decision True, allow a double or a "Surrender"
- If dealer's hand is an "A", allow the player to choose "Insurance"
- If player's hand == card : card (same cards), allow the option to "split"
- Always allow the options to "Hit" or "Stand".

### Apply action

- If Decision == "HIT", Add another card
- If Decision == "STAND", do not add another card
- IF Decision == "DOUBLE", Add only one more card
- If Decision == "SURRENDER", stop.
- If Decision == "INSURANCE", no change 

In [47]:
def generate_actions(state):
    actions = []
    
    if state["busted"] or state.get("done", False):
        return actions

    actions.append("hit")
    actions.append("stand")

    if state.get("first", False):
        if len(state["hand"]) == 2:
            actions.append("double")
            if len(state["hand"]) == 2 and state["hand"][0] == state["hand"][1]:
                actions.append("split")

        actions.append("surrender")

        if state["dealer"] == "A":
            actions.append("insurance")

    return actions
# generate_actions()

In [48]:
def apply_action(state, action, next_card):
    if action == "hit":
        if next_card:
            state["hand"].append(next_card)
            state["total"] = hand_values(state["hand"])
            state["busted"] = state["total"] > 21

    elif action == "stand":
        state["done"] = True

    elif action == "double":
        if next_card:
            state["hand"].append(next_card)
            state["total"] = hand_values(state["hand"])
            state["bet"] = state.get("bet", 1) * 2
            state["done"] = True
            state["busted"] = state["total"] > 21


    elif action == "split":
        # Splitting creates two hands
        card1, card2 = state["hand"]
        state["hands"] = [[card1], [card2]]
        state.pop("hand")  # remove single hand
        state["split"] = True

    elif action == "surrender":
        state["surrendered"] = True
        state["done"] = True

    elif action == "insurance":
        state["insurance"] = True

    return state

In [49]:
state = {
    "hand": ["10", "6"],
    "dealer": "9",
    "first": True,
    "total": 16,
    "busted": False,
    "bet": 1
}

print("Legal actions:", generate_actions(state))
# Suppose player hits and draws a 5
new_state = apply_action(state, "hit", next_card="5")
print("Updated state:", new_state)


Legal actions: ['hit', 'stand', 'double', 'surrender']
Updated state: {'hand': ['10', '6', '5'], 'dealer': '9', 'first': True, 'total': 21, 'busted': False, 'bet': 1}
